In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Tb2Ti2O7 - single-crystal neutron CW - anisotropic ADPs

Verifies calculated F2 values for a no-extinction neutron
single-crystal reference with anisotropic ADPs.

**Refinement:** the overall scale only; all other parameters are
taken from the FullProf reference.

In [2]:
import easydiffraction as edi
from easydiffraction import ExperimentFactory
from easydiffraction import StructureFactory
from easydiffraction.analysis import verification as verify

## Build the project

In [3]:
project = edi.Project()

## Define the structure

In [4]:
structure = StructureFactory.from_scratch(name='tbti')

structure.space_group.name_h_m = 'F d -3 m'  # FullProf Space group symbol

structure.cell.length_a = 10.130  # FullProf a

# Anisotropic sites carry the FullProf beta tensor directly: ``adp_type``
# is set to ``'beta'`` and the dimensionless beta components are assigned
# verbatim. F d -3 m is cubic, so site symmetry links the remaining beta
# components and only the independent ones are set. FullProf occupancy is
# the site multiplicity over the general multiplicity; CIF/EasyDiffraction
# use 1.0 for a fully occupied site.
structure.atom_sites.create(
    id='Tb',  # FullProf Atom
    type_symbol='Tb',  # FullProf Typ
    fract_x=0.5,  # FullProf X
    fract_y=0.5,  # FullProf Y
    fract_z=0.5,  # FullProf Z
    adp_type='beta',  # FullProf beta tensor
)
aniso = structure.atom_site_aniso['Tb']
aniso.adp_11 = 0.00098991673  # FullProf beta11
aniso.adp_12 = -0.00047650724  # FullProf beta12

structure.atom_sites.create(
    id='Ti',  # FullProf Atom
    type_symbol='Ti',  # FullProf Typ
    fract_x=0.0,  # FullProf X
    fract_y=0.0,  # FullProf Y
    fract_z=0.0,  # FullProf Z
    adp_type='beta',  # FullProf beta tensor
)
aniso = structure.atom_site_aniso['Ti']
aniso.adp_11 = 0.00090989727  # FullProf beta11
aniso.adp_12 = -0.00016990340  # FullProf beta12

structure.atom_sites.create(
    id='O1',  # FullProf Atom
    type_symbol='O',  # FullProf Typ
    fract_x=0.32804,  # FullProf X
    fract_y=0.125,  # FullProf Y
    fract_z=0.125,  # FullProf Z
    adp_type='beta',  # FullProf beta tensor
)
aniso = structure.atom_site_aniso['O1']
aniso.adp_11 = 0.0012294180  # FullProf beta11
aniso.adp_22 = 0.00078215479  # FullProf beta22
aniso.adp_23 = 0.00041246481  # FullProf beta23

structure.atom_sites.create(
    id='O2',  # FullProf Atom
    type_symbol='O',  # FullProf Typ
    fract_x=0.375,  # FullProf X
    fract_y=0.375,  # FullProf Y
    fract_z=0.375,  # FullProf Z
    adp_type='beta',  # FullProf beta tensor
)
aniso = structure.atom_site_aniso['O2']
aniso.adp_11 = 0.00060762477  # FullProf beta11

project.structures.add(structure)

In [5]:
structure.show_as_text()

Structure 🧩 'tbti' as text


,Edi
1,data_tbti
2,
3,_cell.length_a 10.13
4,_cell.length_b 10.13
5,_cell.length_c 10.13
6,_cell.angle_alpha 90.
7,_cell.angle_beta 90.
8,_cell.angle_gamma 90.
9,
10,"_space_group.name_h_m ""F d -3 m"""


## Load the FullProf reference

In [6]:
FULLPROF_PROJECT_DIR = 'sc-neut-cwl_tbti_anisotropic-adp'
FULLPROF_OUT_FILE = 'tbti.out'
FULLPROF_SCALE = 4.600  # FullProf Scale
FULLPROF_WAVELENGTH = 0.7930  # FullProf Lambda

f2calc = verify.load_fullprof_sc_f2calc(FULLPROF_PROJECT_DIR, FULLPROF_OUT_FILE)
FULLPROF_LABEL = verify.fullprof_label(FULLPROF_PROJECT_DIR, FULLPROF_OUT_FILE)

## Create the experiment

In [7]:
experiment = ExperimentFactory.from_scratch(
    name='tbti',
    sample_form='single crystal',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
    scattering_type='bragg',
)

experiment.linked_structure.structure_id = 'tbti'
experiment.linked_structure.scale = FULLPROF_SCALE
experiment.instrument.setup_wavelength = FULLPROF_WAVELENGTH

verify.set_reference_reflections(experiment, f2calc)

project.experiments.add(experiment)

## edi-cryspy VS FullProf

In [8]:
calc_ed_cryspy = verify.calculate_reflections(project, experiment, 'cryspy')
LABEL_ED_CRYSPY = verify.engine_label('cryspy')
reference, candidate = verify.align_reflections(f2calc, calc_ed_cryspy)

project.display.reflection_comparison(
    'tbti',
    reference=reference,
    candidate=candidate,
    reference_label=FULLPROF_LABEL,
    candidate_label=LABEL_ED_CRYSPY,
)

Calculator for experiment 'tbti' already set to


cryspy


## Fit edi-cryspy to FullProf

In [9]:
experiment.calculator.type = 'cryspy'

experiment.linked_structure.scale.free = True

project.analysis.fit()
project.display.fit.results()

calc_ed_cryspy_refined = verify.calculate_reflections(project, experiment, 'cryspy')
LABEL_ED_CRYSPY_REFINED = verify.engine_label('cryspy', note='scale only')
reference_refined, candidate_refined = verify.align_reflections(f2calc, calc_ed_cryspy_refined)

project.display.reflection_comparison(
    'tbti',
    reference=reference_refined,
    candidate=candidate_refined,
    reference_label=FULLPROF_LABEL,
    candidate_label=LABEL_ED_CRYSPY_REFINED,
)

verify.report_refinement_closeness(
    reference,
    candidate,
    candidate_refined,
)

Calculator for experiment 'tbti' already set to


cryspy


<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'tbti' for 'single' fitting


...

📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,tbti,linked_structure,,scale,,4.6000,2.3014,0.0001,49.97 % ↓


Calculator for experiment 'tbti' already set to


cryspy


,Metric,Before,After
1,Profile diff (%),99.88,0.05
2,Max deviation (%),99.87,0.08
3,Area ratio,1.9991,1.0001
4,Shape correlation,1.0000,1.0000


## Agreement check

In [10]:
verify.assert_patterns_agree([
    (f'{LABEL_ED_CRYSPY_REFINED} vs {FULLPROF_LABEL}', reference_refined, candidate_refined),
])

,Comparison,Metric,Expected,Actual,OK
1,"edi 0.20.0+dev3 (cryspy 0.13.0, scale only) vs FullProf 8.40",Profile diff (%),< 2.5,0.05,✅
2,,Max deviation (%),< 6,0.08,✅
3,,Area ratio,0.99 to 1.01,1.0001,✅
4,,Shape correlation,> 0.999,1.0000,✅


True